# Lecture: Pandas II

[Acknowledgments Page](https://ds100.org/sp24/acks/)

A demonstration of advanced `pandas` syntax.

In [1]:
# Import numpy, pandas
import numpy as np
import pandas as pd

## Dataset: California baby names

In today's lecture, we'll work with the `babynames` dataset, which contains information about the names of infants born in California.

The cell below pulls census data from a government website and then loads it into a usable form. The code shown here is outside of the scope of CSCI-9, but you're encouraged to dig into it if you are interested!

In [2]:
import urllib.request
import os.path
import zipfile

data_url = "https://www.ssa.gov/oact/babynames/state/namesbystate.zip"
local_filename = "data/babynamesbystate.zip"
if not os.path.exists(local_filename): # If the data exists don't download again
    with urllib.request.urlopen(data_url) as resp, open(local_filename, 'wb') as f:
        f.write(resp.read())

zf = zipfile.ZipFile(local_filename, 'r')

ca_name = 'STATE.CA.TXT'
field_names = ['State', 'Sex', 'Year', 'Name', 'Count']
with zf.open(ca_name) as fh:
    babynames = pd.read_csv(fh, header=None, names=field_names)

babynames.head()

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134


## Conditional Selection

In [3]:
# Select the first 10 rows
# Ask yourself: Why is :9 is the correct slice to select the first 10 rows?
babynames_first_10_rows = babynames.loc[:9, :]

babynames_first_10_rows

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134
5,CA,F,1910,Ruth,128
6,CA,F,1910,Evelyn,126
7,CA,F,1910,Alice,118
8,CA,F,1910,Virginia,101
9,CA,F,1910,Elizabeth,93


By passing in a sequence (list, array, or `Series`) of boolean values, we can extract a subset of the rows in a `DataFrame`. We will keep *only* the rows that correspond to a boolean value of `True`.

In [4]:
# Pass in a sequence of boolean values
# Notice how we have exactly 10 elements in our boolean array argument
babynames_first_10_rows[[True, False, True, False, True, False, True, False, True, False]]

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
2,CA,F,1910,Dorothy,220
4,CA,F,1910,Frances,134
6,CA,F,1910,Evelyn,126
8,CA,F,1910,Virginia,101


In [5]:
# Or use .loc to filter a DataFrame by a Boolean array argument.
babynames_first_10_rows.loc[[True, False, True, False, True, False, True, False, True, False], :]

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
2,CA,F,1910,Dorothy,220
4,CA,F,1910,Frances,134
6,CA,F,1910,Evelyn,126
8,CA,F,1910,Virginia,101


Oftentimes, we'll use boolean selection to check for entries in a `DataFrame` that meet a particular condition.

In [6]:
# First, use a logical condition to generate a boolean Series
logical_operator = (babynames["Sex"] == "F")
logical_operator

0          True
1          True
2          True
3          True
4          True
          ...  
407423    False
407424    False
407425    False
407426    False
407427    False
Name: Sex, Length: 407428, dtype: bool

In [7]:
# Then, use this boolean Series to filter the DataFrame
babynames[logical_operator]

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134
...,...,...,...,...,...
239532,CA,F,2022,Zemira,5
239533,CA,F,2022,Ziggy,5
239534,CA,F,2022,Zimal,5
239535,CA,F,2022,Zosia,5


Boolean selection also works with `loc`!

In [8]:
# Notice that we did not have to specify columns to select 
# If no columns are referenced, pandas will automatically select all columns
babynames.loc[babynames["Sex"] == "F"]

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134
...,...,...,...,...,...
239532,CA,F,2022,Zemira,5
239533,CA,F,2022,Ziggy,5
239534,CA,F,2022,Zimal,5
239535,CA,F,2022,Zosia,5


To filter on multiple conditions, we combine boolean operators using **bitwise comparisons**.

Symbol | Usage      | Meaning 
------ | ---------- | -------------------------------------
~    | ~p       | Returns negation of p
&#124; | p &#124; q | p OR q
&    | p & q    | p AND q
^  | p ^ q | p XOR q (exclusive or)

In [9]:
# Filter for baby names that are female AND popular before 2000
babynames[(babynames["Sex"] == "F") & (babynames["Year"] < 2000)]

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134
...,...,...,...,...,...
149050,CA,F,1999,Zareen,5
149051,CA,F,1999,Zeinab,5
149052,CA,F,1999,Zhane,5
149053,CA,F,1999,Zoha,5


In [10]:
# Filter for baby names that are female OR popular before 2000
babynames[(babynames["Sex"] == "F") | (babynames["Year"] < 2000)]

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134
...,...,...,...,...,...
342435,CA,M,1999,Yuuki,5
342436,CA,M,1999,Zakariya,5
342437,CA,M,1999,Zavier,5
342438,CA,M,1999,Zayn,5


In [11]:
# .isin
names = ["Audrey", "Imaginary Friend", "Bob"]
babynames[babynames["Name"].isin(names)]

,State,Sex,Year,Name,Count
155,CA,F,1910,Audrey,8
335,CA,F,1911,Audrey,16
598,CA,F,1912,Audrey,22
905,CA,F,1913,Audrey,24
1222,CA,F,1914,Audrey,31
...,...,...,...,...,...
368946,CA,M,2009,Bob,7
375309,CA,M,2011,Bob,5
380795,CA,M,2013,Bob,6
386442,CA,M,2015,Bob,7


In [12]:
# .str.startswith
babynames[babynames["Name"].str.startswith("A")]

,State,Sex,Year,Name,Count
7,CA,F,1910,Alice,118
31,CA,F,1910,Anna,51
32,CA,F,1910,Ann,47
57,CA,F,1910,Anita,28
68,CA,F,1910,Alma,24
...,...,...,...,...,...
407112,CA,M,2022,Avinash,5
407113,CA,M,2022,Avyukth,5
407114,CA,M,2022,Aycen,5
407115,CA,M,2022,Ayrton,5


## Adding, Removing, and Modifying Columns

To add a column, use `[]` to reference the desired new column, then assign it to a `Series` or array of appropriate length.

In [13]:
# Create a Series of the length of each name
babyname_lengths = babynames["Name"].str.len()

# Add a column named "name_lengths" that includes the length of each name
babynames["name_lengths"] = babyname_lengths

babynames

,State,Sex,Year,Name,Count,name_lengths
0,CA,F,1910,Mary,295,4
1,CA,F,1910,Helen,239,5
2,CA,F,1910,Dorothy,220,7
3,CA,F,1910,Margaret,163,8
4,CA,F,1910,Frances,134,7
...,...,...,...,...,...,...
407423,CA,M,2022,Zayvier,5,7
407424,CA,M,2022,Zia,5,3
407425,CA,M,2022,Zora,5,4
407426,CA,M,2022,Zuriel,5,6


To modify a column, use `[]` to access the desired column, then re-assign it to a new array or Series.

In [14]:
# Modify the "name_lengths" column to be one less than its original value
babynames["name_lengths"] = babynames["name_lengths"] - 1
babynames

,State,Sex,Year,Name,Count,name_lengths
0,CA,F,1910,Mary,295,3
1,CA,F,1910,Helen,239,4
2,CA,F,1910,Dorothy,220,6
3,CA,F,1910,Margaret,163,7
4,CA,F,1910,Frances,134,6
...,...,...,...,...,...,...
407423,CA,M,2022,Zayvier,5,6
407424,CA,M,2022,Zia,5,2
407425,CA,M,2022,Zora,5,3
407426,CA,M,2022,Zuriel,5,5


Rename a column using the `.rename()` method.

In [15]:
# Rename "name_lengths" to "Length"
babynames = babynames.rename(columns={"name_lengths":"Length"})
babynames

,State,Sex,Year,Name,Count,Length
0,CA,F,1910,Mary,295,3
1,CA,F,1910,Helen,239,4
2,CA,F,1910,Dorothy,220,6
3,CA,F,1910,Margaret,163,7
4,CA,F,1910,Frances,134,6
...,...,...,...,...,...,...
407423,CA,M,2022,Zayvier,5,6
407424,CA,M,2022,Zia,5,2
407425,CA,M,2022,Zora,5,3
407426,CA,M,2022,Zuriel,5,5


Remove a column using `.drop()`.

In [16]:
# Remove our new "Length" column
babynames = babynames.drop("Length", axis="columns")
babynames

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134
...,...,...,...,...,...
407423,CA,M,2022,Zayvier,5
407424,CA,M,2022,Zia,5
407425,CA,M,2022,Zora,5
407426,CA,M,2022,Zuriel,5


In [38]:
babynames

,State,Sex,Year,Name,Count
115957,CA,F,1990,Deandrea,5
101976,CA,F,1986,Deandrea,6
131029,CA,F,1994,Leandrea,5
108731,CA,F,1988,Deandrea,5
308131,CA,M,1985,Deandrea,6
...,...,...,...,...,...
202350,CA,F,2013,Lailah,50
212784,CA,F,2015,Helene,6
343933,CA,M,2000,Kelton,9
212787,CA,F,2015,Holley,6


## Useful Utility Functions

#### `NumPy`

The `NumPy` functions you encountered in [CSCI-8](https://www.data8.org/su23/reference/#array-functions-and-methods) are compatible with objects in `pandas`. 

In [17]:
# Number of babies named Yash each year
yash_counts = babynames[babynames["Name"] == "Yash"]["Count"]
yash_counts

331824     8
334114     9
336390    11
338773    12
341387    10
343571    14
345767    24
348230    29
350889    24
353445    29
356221    25
358978    27
361831    29
364905    24
367867    23
370945    18
374055    14
376756    18
379660    18
383338     9
385903    12
388529    17
391485    16
394906    10
397874     9
400171    15
403092    13
406006    13
Name: Count, dtype: int64

In [18]:
# Average number of babies named Yash each year
np.mean(yash_counts)

17.142857142857142

In [19]:
# Max number of babies named Yash born in any single year
max(yash_counts)

29

#### Built-In `pandas` Methods

There are many, *many* utility functions built into `pandas`, far more than we can possibly cover in lecture. You are encouraged to explore all the functionality outlined in the `pandas` [documentation](https://pandas.pydata.org/docs/reference/index.html).

In [20]:
# Return the shape of the object in the format (num_rows, num_columns)
babynames.shape

(407428, 5)

In [21]:
# Return the total number of entries in the object, equal to num_rows * num_columns
babynames.size

2037140

In [22]:
# What summary statistics can we describe?
babynames.describe()

,Year,Count
count,407428.000000,407428.000000
mean,1985.733609,79.543456
std,27.007660,293.698654
min,1910.000000,5.000000
25%,1969.000000,7.000000
50%,1992.000000,13.000000
75%,2008.000000,38.000000
max,2022.000000,8260.000000


In [23]:
# Our statistics are slightly different when working with a Series
babynames["Sex"].describe()

count     407428
unique         2
top            F
freq      239537
Name: Sex, dtype: object

In [24]:
# Randomly sample row(s) from the DataFrame
babynames.sample()

,State,Sex,Year,Name,Count
266524,CA,M,1955,Joaquin,16


In [25]:
# Randomly sample 5 rows and select results from column position 2
# Rerun this cell a few times – you'll get different results!
babynames.sample(5).iloc[:, 2:]

,Year,Name,Count
204944,2013,Olina,6
366998,2009,Tristan,545
173982,2006,Osiris,20
14053,1935,Eleanore,9
49893,1964,Clara,44


In [26]:
# Sampling with replacement
babynames[babynames["Year"] == 2000].sample(4, replace = True).iloc[:,2:]

,Year,Name,Count
149373,2000,Bryanna,136
343874,2000,Abdulrahman,9
343865,2000,Thaddeus,10
343054,2000,Kian,36


In [27]:
# Count the number of times each unique value occurs in a Series
babynames["Name"].value_counts()

Name
Jean         223
Francis      221
Guadalupe    218
Jessie       217
Marion       214
            ... 
Renesme        1
Purity         1
Olanna         1
Nohea          1
Zayvier        1
Name: count, Length: 20437, dtype: int64

In [28]:
# Return an array of all unique values in the Series
babynames["Name"].unique()

array(['Mary', 'Helen', 'Dorothy', ..., 'Zae', 'Zai', 'Zayvier'],
      dtype=object)

In [29]:
# Sort a Series
babynames["Name"].sort_values()

366001      Aadan
384005      Aadan
369120      Aadan
398211    Aadarsh
370306      Aaden
           ...   
220691      Zyrah
197529      Zyrah
217429      Zyrah
232167      Zyrah
404544      Zyrus
Name: Name, Length: 407428, dtype: object

In [30]:
# Sort a DataFrame – there are lots of Michaels in California
babynames.sort_values(by="Count", ascending=False)

,State,Sex,Year,Name,Count
268041,CA,M,1957,Michael,8260
267017,CA,M,1956,Michael,8258
317387,CA,M,1990,Michael,8246
281850,CA,M,1969,Michael,8245
283146,CA,M,1970,Michael,8196
...,...,...,...,...,...
317292,CA,M,1989,Olegario,5
317291,CA,M,1989,Norbert,5
317290,CA,M,1989,Niles,5
317289,CA,M,1989,Nikola,5


## Custom sorting

### Approach 1: Create a temporary column

In [31]:
# Create a Series of the length of each name
babyname_lengths = babynames["Name"].str.len()

# Add a column named "name_lengths" that includes the length of each name
babynames["name_lengths"] = babyname_lengths
babynames.head(5)

,State,Sex,Year,Name,Count,name_lengths
0,CA,F,1910,Mary,295,4
1,CA,F,1910,Helen,239,5
2,CA,F,1910,Dorothy,220,7
3,CA,F,1910,Margaret,163,8
4,CA,F,1910,Frances,134,7


In [32]:
# Sort by the temporary column
babynames = babynames.sort_values(by="name_lengths", ascending=False)
babynames.head(5)

,State,Sex,Year,Name,Count,name_lengths
334166,CA,M,1996,Franciscojavier,8,15
337301,CA,M,1997,Franciscojavier,5,15
339472,CA,M,1998,Franciscojavier,6,15
321792,CA,M,1991,Ryanchristopher,7,15
327358,CA,M,1993,Johnchristopher,5,15


In [33]:
# Drop the 'name_length' column
babynames = babynames.drop("name_lengths", axis="columns")
babynames.head(5)

,State,Sex,Year,Name,Count
334166,CA,M,1996,Franciscojavier,8
337301,CA,M,1997,Franciscojavier,5
339472,CA,M,1998,Franciscojavier,6
321792,CA,M,1991,Ryanchristopher,7
327358,CA,M,1993,Johnchristopher,5


### Approach 2: Sorting using the `key` argument

In [34]:
# Sort using the key argument
babynames.sort_values("Name", key=lambda x:x.str.len(), ascending=False).head()

,State,Sex,Year,Name,Count
334166,CA,M,1996,Franciscojavier,8
327472,CA,M,1993,Ryanchristopher,5
337301,CA,M,1997,Franciscojavier,5
337477,CA,M,1997,Ryanchristopher,5
312543,CA,M,1987,Franciscojavier,5


In [35]:
# Write a separate function instead of using lambda function
def myFunc(x):
    return x.str.len()

babynames.sort_values("Name", key=myFunc, ascending=False).head()

,State,Sex,Year,Name,Count
334166,CA,M,1996,Franciscojavier,8
327472,CA,M,1993,Ryanchristopher,5
337301,CA,M,1997,Franciscojavier,5
337477,CA,M,1997,Ryanchristopher,5
312543,CA,M,1987,Franciscojavier,5


### Approach 3: Sorting Using the `map` Function

We can also use the Python map function if we want to use an arbitrarily defined function. Suppose we want to sort by the number of occurrences of "dr" plus the number of occurences of "ea".

In [36]:
# First, define a function to count the number of times "dr" or "ea" appear in each name
def dr_ea_count(string):
    return string.count('dr') + string.count('ea')

# Then, use `map` to apply `dr_ea_count` to each name in the "Name" column
babynames["dr_ea_count"] = babynames["Name"].map(dr_ea_count)

# Sort the DataFrame by the new "dr_ea_count" column so we can see our handiwork
babynames = babynames.sort_values(by="dr_ea_count", ascending=False)
babynames.head()

,State,Sex,Year,Name,Count,dr_ea_count
115957,CA,F,1990,Deandrea,5,3
101976,CA,F,1986,Deandrea,6,3
131029,CA,F,1994,Leandrea,5,3
108731,CA,F,1988,Deandrea,5,3
308131,CA,M,1985,Deandrea,6,3


In [37]:
# Drop the `dr_ea_count` column
babynames = babynames.drop("dr_ea_count", axis="columns")
babynames.head(5)

,State,Sex,Year,Name,Count
115957,CA,F,1990,Deandrea,5
101976,CA,F,1986,Deandrea,6
131029,CA,F,1994,Leandrea,5
108731,CA,F,1988,Deandrea,5
308131,CA,M,1985,Deandrea,6
